# 리포트 00 — 기초: Sionna 로 되는 것과, 표적 산란이 시작되는 자리

> ### 한 일
> **Sionna RT 설치본을 인자 목록까지 해부해 광선이 면을 맞았을 때 무엇이 계산되는지를 적고, 표적 산란이 어디서부터 별도 항이 되는지를 실측과 결정표로 갈랐다.**

### 결과
1. Sionna 는 자기 문제를 정확히 푼다 — 자유공간 직접파가 Friis 이론과 6.3e-07 dB ⟨outputs/report00_evidence.json : F_what_sionna_gets_right.numbers.los_agreement_db⟩ 안이고, 평면 반사 진폭이 이미지-소스 해석해 대비 0.9997 ⟨outputs/report00_sionna_probe.json : exp_c_spreading_check.ratio_measured_over_predicted⟩ 다.
2. 금속 평판의 면적을 1600 ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.area_ratio_max⟩배 키우면 PO 단면적은 64.08 dB ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.po_theory_span_db⟩ 커지고, path solver 의 표적 경로 진폭은 7.4e-07 dB ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.rt_span_db⟩ 움직인다 — 경로 수는 전 구간 1 ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.n_paths_target_set_union[0]⟩개다.
3. 같은 재질·같은 정면면적에서 모양만 바꾸면 σ 가 31.29 dB ⟨outputs/report00_evidence.json : C_same_material_different_shape.numbers.shape_gap_db⟩ 갈린다 — 같은 반사계수 위에서 위상 정렬이 답을 정한다.
4. 우리 PO 커널은 해석 PO 대비 0.201 dB ⟨outputs/report00_po_case.json : s3_validation.layer1_analytic_po_convergence.kr_sweep_max_abs_db_vs_po_div16⟩ 안에서 수렴하고, 유효 하한은 특징 폭 0.729 ⟨outputs/report00_po_case.json : s4_limits.po_validity_knee_a_over_lambda⟩λ 다.
5. 결정표는 두 문장으로 갈린다 — 표적 항이 비에서 상수로 소거되는가, 절대값이 필요한가(그림 4).

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 경로 탐색·필드 계산의 인자 | 설치본 2.0.1 ⟨outputs/report00_sionna_anatomy.json : item6_versions.values.sionna_rt⟩ 소스를 직접 읽고 `inspect.signature` 로 런타임 재확인 |
| 크기 무응답 | 빈 씬에서 평판 변만 바꾸며 예산 4단 · 시드 5개 · 깊이 2단으로 24 ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.n_cells⟩셀 반복 |
| 우리 커널의 정확도 | 해석 PO 구 · PEC 구 Mie 정확해 · 얇은 띠 2D EFIE MoM · PEC 이면각 닫힌형과 대조 |
| 절대 레벨 | 공개 실측(Das, IEEE WCL 2026)에 A(f) 의 기울기를 맞춤 |
| 결정표의 배치 | 판단이다. 행마다 그 판단이 선 근거 JSON 키를 붙였다 |

### 재현

```bash
PYTHONPATH=src python benchmark/build_report00_po_case.py
PYTHONPATH=src python benchmark/build_report00_decision_map.py
PYTHONPATH=src python src/figs_report00.py
PYTHONPATH=src python src/make_report00_foundations.py
```

| | |
|---|---|
| 출력 | `outputs/report00_po_case.json`, `outputs/report00_decision_map.json`, `outputs/report00_sionna_anatomy.json`, `outputs/report00_sionna_probe.json`, `outputs/report00_evidence.json` |
| 소요 | 약 3분 (GPU 0장 — 전부 JSON 읽기와 그림 그리기다) |
| 비고 | anatomy · probe · evidence 세 JSON 은 설치본 해부와 실행 프로브의 산출물이고, 각 파일의 `_meta` 가 자기 생성기 경로를 들고 있다 |

---

## §1. Sionna 는 광선이 맞으면 무엇을 계산하는가

답은 **두 단계**로 만들어진다. **① 경로 탐색.** 소스마다 광선을 구면에 뿌려(피보나치 격자, 난수가 아니다) 어느 면들을 어떤 순서로 맞는지 후보를 모은다.

같은 면 순서를 발견한 광선은 **첫 발만 남기고 나머지를 버린다** — 면 해시로 만든 경로 지문의 카운터를 원자적으로 올리고 `samples_counter == 0` 인 광선만 저장한다(`sb_candidate_generator.py:484-498`). 그다음 이미지법이 소스를 각 면에 거울반사시켜 교점 좌표를 해석적으로 다시 푼다(`image_method.py:37-47`).

**② 필드 계산.** 그렇게 확정된 경로 **하나**를 따라가며 진폭 a 를 만든다(`field_calculator.py`).

그래서 광선은 **정찰병**이고, 답의 단위는 경로다. 이 구분이 §2 의 출발점이다.

그림을 셋 잡고 들어간다. 비유는 **깨지는 자리**를 함께 적어야 쓸모가 있다 (근거 `outputs/report00_po_case.json:s0_fair_boundary.analogies_and_where_they_break`).

| 비유 | 그래서 맞는 것 | ⚠ 이 비유가 깨지는 자리 |
|---|---|---|
| Sionna 의 표면 처리는 **거울 한 장**이다 | 반사 세기(Fresnel)도 방향도 맞다 | 거울은 각도만 돌려준다 — 평판을 키워도 진폭이 그대로인 것이 그 뜻이다(§3) |
| PO 표면적분은 **조명면에 붙은 작은 안테나들의 합**이다 | 점마다 위상이 더해지므로 모양이 바뀌면 보강·상쇄가 바뀐다 — σ 가 여기서 창발한다 | 그 작은 안테나의 세기를 국소 평면 반사로 정한다. 특징 폭이 파장 아래로 가면 그 가정이 깨진다(§6 의 무릎) |
| SBR 은 **손전등으로 비추고 빛이 닿은 자리만 세는 것**이다 | 자기가림이 공짜로 처리된다 — 빛이 닿은 면만 세면 된다 | 손전등은 모서리에서 휘는 빛과 몸통을 감아 도는 빛을 빼놓는다 — 전자가 PTD, 후자가 크리핑파다(§8) |

### Sionna 가 정확히 계산하는 것 — 먼저 이것부터

| 메커니즘 | 코드 | 무엇을 어떻게 |
|---|---|---|
| 정반사 세기 | radio_material.py:560-562, 853-892 | ITU-R P.2040 단층 슬래브 Fresnel r_te/r_tm 을 정확히 구현. 편파는 Jones 행렬로 완전히 처리. |
| 투과(굴절) | path_solver.py:153 (기본 True) | 두께 d 를 반영한 슬래브 투과계수. 단 광선은 꺾이지 않고 직진(얇은 벽 가정, path_solver.py:36-41 이 명시). |
| 가림·그림자 | Mitsuba 광선-삼각형 교차 + image_method.py:41-47 역추적 검증 | 유한 기하로 정확히 판정. 표적이 벽 뒤에 있으면 제대로 사라진다. |
| 1차 UTD 쐐기 회절 | radio_material.py:964-1144 | Kouyoumjian-Pathak + Luebbers 유한도전율. 기본값 off 일 뿐 구현은 완비. |
| 다중 반사·기하 | path_solver.py:146 max_depth=3 | 임의 순서의 반사/투과/확산 조합 경로. |
| 지연·도플러 | field_calculator.py:355, 526- | τ = 경로길이/c. 도플러는 객체당 강체 속도 1벡터 기준. |
| 확산산란(경험모델) | radio_material.py:914-962 | 거친 표면의 에너지 분산. S 로 정반사와 배분. 단 기본 S=0. |

출처 ⟨outputs/report00_sionna_anatomy.json : item9_verdict.can_do⟩

### 필드 계산의 식 — 경로 하나가 진폭 하나가 되는 자리

`a = (안테나 패턴) × (경로 위 Jones 행렬들의 곱) × spreading_factor × λ/4π`

기호를 하나씩. **Jones 행렬**은 전계를 (⊥, ∥) 2성분 복소 벡터로 보고 그것을 변환하는 2×2 행렬이고, 정반사에서는 대각 성분이 Fresnel 계수 r_te · r_tm 이다. **spreading_factor** 는 파면이 퍼지면서 진폭이 줄어드는 비율 [1/m] 이고, **λ/4π** 는 등방 안테나의 유효개구 λ²/4π 를 진폭 차원으로 옮긴 상수 [m] 다.

반사 경로의 spreading_factor 는 `1/ray_tube_length` 하나로 끝난다(`field_calculator.py:323-326`). 왜 거리만으로 끝나는지가 이 편의 첫 유도다.

점원에서 나온 구면파는 진폭이 1/r 로 준다. 이 파가 **평평한** 면에 부딪히면 반사파는 여전히 구면파이고 그 중심은 소스를 면에 대해 거울반사시킨 상(image)이다. 상은 면 뒤 s′ 에 있으므로 반사점에서 s 를 더 간 수신점은 상으로부터 s′+s 이고, 진폭은 1/(s′+s) — 정확히 `1/ray_tube_length` 다.

### 굽은 면이면 무엇이 더 붙는가 — 그리고 회절과 λ 의 자리

일반 기하광학의 광선관 확산인자는 `A(s) = sqrt( ρ₁ρ₂ / ((ρ₁+s)(ρ₂+s)) )` 이고, ρ₁·ρ₂ 는 반사 **직후** 파면의 두 주곡률반경이다. 곡면 반사에서 이 ρ 는 입사 파면의 곡률과 **면의 주곡률**이 섞여 정해진다.

면이 평평하면 면곡률 항이 0 이 되어 ρ = s′ 가 되고, 위 식이 s′/(s′+s) 로 붕괴한다 — 여기에 소스에서 반사점까지의 1/s′ 를 곱하면 1/(s+s′) 다. **Sionna 는 이 일반 공식의 면곡률 0 특수해를 정확히 구현한다.** 면곡률 항을 채우려면 그 점에서 표면의 제2기본형식이 필요한데, 삼각형 메쉬는 면마다 곡률이 정의상 0 이라 그 항이 구성상 0 으로 남는다.

회절 경로는 `1/sqrt(s·s′·(s+s′))` 로 갈라진다. 두 조각으로 읽으면 뜻이 보인다 — `(1/s′) × sqrt( s′/(s(s+s′)) )` 에서 앞은 모서리까지 오는 구면파의 평범한 확산이고, 뒤가 UTD 표준 모서리 인자다. sqrt 가 붙는 이유는 모서리에서 나온 파가 켈러 원뿔을 따라 한 방향으로 **원통형**으로 퍼지기 때문이다.

⚠ 마지막 λ/4π 를 보고 '파장이 들어가니 산란도 다루겠지' 라고 읽기 쉽다. 그 λ 는 **수신 안테나** 항이고, 표적의 전기적 크기 L/λ 는 이 식 밖에 있다.

![report00 f1](outputs/figures/report00_f1.png)

**그림 1.** 같은 광선엔진을 쓰는 두 계산은 표면에서 무엇이 갈라지는가?

광선 수는 이 진폭에 들어가지 않는다 — 실행으로 확인했다. 10,000 ⟨outputs/report00_sionna_probe.json : exp_a_ray_count_sweep[0].samples_per_src⟩ 발에서 4,000,000 ⟨outputs/report00_sionna_probe.json : exp_a_ray_count_sweep[-1].samples_per_src⟩ 발까지 400 ⟨outputs/report00_sionna_probe.json : summary.ray_count_span⟩배를 올려도 정반사 진폭 스프레드는 0.0 dB ⟨outputs/report00_sionna_probe.json : summary.abs_a_spread_over_ray_count_db⟩ 다.

⭐ 공정하게 단서를 붙인다. 광선 수가 진폭에 **전혀** 안 들어가는 것은 정반사·투과·회절 경로에서 참이다.

확산산란 경로에서는 다르다 — `solid_angle` 이 4π/N 으로 초기화돼 재질까지 실려 가고, 진폭에 sqrt(fs · solid_angle) 로 곱해져 1/sqrt(N) 으로 스케일된다.

그것이 몬테카를로 추정량의 올바른 정규화다 — 경로 하나는 작아지고 경로 개수가 N 에 비례해 늘어 총 전력이 수렴한다. 그 4π/N 은 **첫 상호작용에만** 살아 있고, 확산이 샘플링되는 순간 2π(반구 입체각)로 덮어써진다(근거 `item1_ray_shooting_and_dedup.d_ray_count_in_amplitude`).

## §2. 그 수식에 없는 것 — 인자 여덟 개를 그대로 센다

| 인자 | 무엇인가 |
|---|---|
| to_world | 국소→월드 3x3 회전 행렬 (자세, 크기 아님) |
| ki_local | 입사 진행 방향 단위벡터 |
| ko_local | 산란 진행 방향 단위벡터 |
| reflection | 반사인가 투과인가 하는 불리언 |
| r_te | TE 반사계수 (Fresnel, 복소) |
| r_tm | TM 반사계수 (Fresnel, 복소) |
| t_te | TE 투과계수 (복소) |
| t_tm | TM 투과계수 (복소) |

출처 ⟨outputs/report00_sionna_anatomy.json : item2_field_calculation_arguments.argument_inventory⟩

### 목록 밖에 있는 양들

| 필드 갱신 인자 목록 밖에 있는 양 |
|---|
| 표면 곡률 (주곡률반경 R1, R2) |
| 물체 면적 A |
| 물체 치수 L (전장·폭·높이) |
| 부딪힌 삼각형의 면적 또는 변 길이 |
| 삼각형 정점 좌표 (return_vertices=False 로 명시적으로 요청하지 않음) |
| 인접 삼각형 정보 (회절을 끈 경우) |
| 파장 (이 함수 안에는 없음; 상위 Fresnel 계수 계산에만 들어감) |

출처 ⟨outputs/report00_sionna_anatomy.json : item2_field_calculation_arguments.absent_quantities⟩

여덟 인자 중 여섯은 방향(단위벡터·회전)이고 둘은 Fresnel 계수다. 더 결정적인 것은 호출부다 — `field_calculator.py:404-405` 가 `return_vertices=False` 로 **정점 좌표를 일부러 요청하지 않고** 법선만 가져온다. 삼각형 크기를 알 수 있는 유일한 통로가 그 자리에서 닫힌다.

### 정확히 말하는 법 — '무한평면 가정' 은 거친 요약이다

Sionna 가 무한평면을 가정한다고 쓰면 반박당한다. 기하는 유한하고, 광선이 그 삼각형을 맞았는지는 Mitsuba 가 정확히 판정한다(가림·그림자는 제대로 작동한다).

정확한 진술은 이것이다 — **광선이 면을 맞았는가는 유한 기하로 판정하고, 맞은 뒤 필드를 얼마나 바꿀지는 국소 평면파–평면경계 문제의 해로 계산한다.** 즉 크기는 `yes/no` 에만 쓰인다. `how much` 는 국소 해가 정한다.

기술보고서 원문이 그 층을 각각 못 박는다.

· 경로 기하 — p.19 — "the image method assumes all reflection surfaces extend infinitely, making the exact in-plane position of a primitive irrelevant to the path geometry" ⟨outputs/report00_evidence.json : G_exact_wording_infinite_surface.numbers.quote_path_geometry⟩

· 계수 — p.46 — "The reflection and refraction coefficients described above assume that the object reflecting the wave or allowing it to penetrate is of infinite size (or thickness)." ⟨outputs/report00_evidence.json : G_exact_wording_infinite_surface.numbers.quote_coefficients⟩

· ⚠ 낱말 주의 — 기술보고서의 'locally planar' 는 **파(wave)** 에 붙는 말이다(p.50 “an incoming locally planar linearly polarized wave”). 표면에 대해서는 조건 없이 extend infinitely · of infinite size 라고 쓴다 (근거 `outputs/report00_evidence.json:G_exact_wording_infinite_surface.numbers.caution_locally`).

### 단위가 이미 답을 말한다 — Γ 는 무차원, σ 는 m²

반사계수 Γ 는 무차원이고 레이더단면적 σ 의 단위는 m² 다. 무차원을 아무리 정확히 계산해도 결과는 무차원으로 남는다. 면적은 **조명면 위의 면적분**에서만 들어온다.

평판의 PO 공식 `σ = 4πA²/λ²` 는 `4π·[m²]²/[m]² = m²` 로 닫히는데, Sionna 의 정반사 진폭 `|a| = λ/(4π(R₁+R₂))` 는 `[m]/[m] = 1` 로 닫힌다 — 면적 기호는 그 식 밖에 있다.

그래서 같은 PEC, 같은 정면면적 0.7854 ⟨outputs/report00_evidence.json : C_same_material_different_shape.numbers.frontal_area_m2⟩ m², 같은 5G 밴드 3.5 GHz ⟨outputs/report00_po_case.json : s4_limits.our_production_bands_vs_knee.nr_ghz⟩ 에서 구는 -1.05 dBsm ⟨outputs/report00_evidence.json : C_same_material_different_shape.numbers.sphere_sigma_dbsm⟩, 평판은 30.24 dBsm ⟨outputs/report00_evidence.json : C_same_material_different_shape.numbers.plate_same_area_sigma_dbsm⟩ 다.

주파수를 두 배로 올리면 평판은 +6.02 dB ⟨outputs/report00_evidence.json : C_same_material_different_shape.numbers.plate_sigma_df_db_per_octave⟩/옥타브, 구는 +0.00 dB ⟨outputs/report00_evidence.json : C_same_material_different_shape.numbers.sphere_sigma_df_db_per_octave⟩/옥타브 움직인다. 갈라지는 이유는 하나다 — 두 값 모두 |Γ|=1 을 쓴다. 차이는 위상이 면 위에서 어떻게 정렬되는가다 — 평판은 A 전체가 같은 위상으로 더해지고(∝A), 구는 곡률이 위상을 흩어 실효 기여면이 λ 규모의 정반사점 근방으로 줄어든다. ⟨outputs/report00_evidence.json : C_same_material_different_shape.formula.why_they_differ⟩

![report00 f3](outputs/figures/report00_f3.png)

**그림 3.** 같은 재질·같은 정면면적에서 모양만 바꾸면 σ 는 얼마나 갈라지는가?

## §3. 실측 — 크기를 흔들어 보면 무엇이 움직이는가

빈 자유공간에 금속 평판 하나를 두고 변만 0.1 m ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.side_m[0]⟩ → 4.0 m ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.side_m[-1]⟩ (면적 1600 ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.area_ratio_max⟩배) 로 키운다. PO 단면적은 면적 dB 당 2.00 dB ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.slope_sigma_db_per_area_db⟩씩, 총 64.08 dB ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.po_theory_span_db⟩ 오른다.

같은 구간에서 path solver 의 표적 경로 진폭은 7.4e-07 dB ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.rt_span_db⟩ 움직이고 경로 수는 내내 1 ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.n_paths_target_set_union[0]⟩개다 — 예산 4단 · 시드 5개 · 깊이 2단 24 ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.n_cells⟩셀 전부에서.

⭐ 그 진폭이 무엇인지도 같이 확인된다. RT 값은 이미지-소스 해석해와 0.0017 dB ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.rt_minus_image_source_max_abs_db⟩ 안에서 맞는다. **엔진은 자기가 푸는 문제를 정확히 푼다.**

두 곡선이 만나는 자리는 변 0.93 m ⟨outputs/report00_evidence.json : A_plate_size_sweep.numbers.side_m_where_rt_equals_po⟩ 한 점뿐이고, 그것은 우연이다.

![report00 f2](outputs/figures/report00_f2.png)

**그림 2.** 표적을 키우면 무엇이 움직이고 무엇이 그대로인가?

### 드론 메쉬에서는 정반사 경로가 자세 하나에서만 살아남는다

같은 실험을 기체 메쉬로 옮긴다. 삼각형을 29,932 ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.n_tri_per_level[0]⟩개 → 450 ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.n_tri_per_level[-1]⟩개(1.82 ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.n_tri_span_decades⟩ decade)로 깎아도 실루엣은 유지된다. 그런데 36 ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.spec_n_aspects⟩자세 중 정반사 경로가 존재하는 자세는 1 ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.spec_n_aspects_nonzero_per_level[0]⟩개다.

그 한 자세에서 진폭은 기여 면 개수를 따라 계단으로 떨어져 총 49.02 dB ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.total_collapse_db⟩ 무너진다 — 면 2→1 계단이 -6.05 dB ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.step_2to1_facet_db⟩ 로 닫힌형 20·log₁₀(1/2) = -6.02 dB ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.theoretical_step_two_to_one_facet_db⟩ 에 붙는다.

⚠ 나머지 자세가 빈 이유는 따로 있다. 같은 자세에 확산을 켜면 표적경유 경로가 자세당 102 ⟨outputs/report00_evidence.json : B_facet_count_sweep.numbers.hot_n_paths_min⟩개 넘게 잡힌다. 비어 있는 것은 광선이 아니라 **거울 조건을 만족하는 삼각형**이다.

### 반대 방향도 같은 뿌리 — 쪼개기만 해도 답이 부푼다

같은 1 m ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.per_side[1].side_m⟩ 평판을 2 ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.per_side[1].n_tri[0]⟩개 → 512 ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.per_side[1].n_tri[-1]⟩개 삼각형으로 **쪼개기만** 해도 코히어런트 전력이 9.54 dB ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.max_inflation_db⟩ 부푼다. 늘어난 경로들은 진폭 산포 0.0 dB ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.duplicate_path_forensics[0].amp_spread_db⟩ · 지연 산포 0.0 ns ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.duplicate_path_forensics[0].tau_spread_ns⟩ · 위상 산포 0.0° ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.duplicate_path_forensics[0].phase_spread_deg⟩ 인 **완전한 복사본**이고, 합은 20·log₁₀(N) 을 0.0017 dB ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.coherent_N_law_max_resid_db⟩ 안에서 따른다. 격자 위치를 옮겨도 중복은 남는다(`offset_removes_duplication` = 아니오 ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.offset_removes_duplication⟩).

메쉬를 깎으면 무너지고 쪼개면 부푼다. 두 방향이 같은 뿌리에서 나온다 — **답의 단위가 면적이 아니라 경로라서** 면 수가 곧 답이 된다.

## §4. 할 수 있는 것과 없는 것 — 결정표

판별 기준은 두 문장이다.

**① 표적 산란량이 비(比)를 취할 때 상수로 소거되는가.** 소거되면 표적 모델 없이도 답이 선다. **② 결론이 절대 dBsm·dB 를 인쇄해야 하는가.**

이 두 물음이 실험을 네 칸으로 가른다.

| 칸 | 표적 항이 소거되는가 | 절대값이 필요한가 | 그래서 무엇을 쓰나 |
|---|---|---|---|
| Z1 | 예 | 아니오 | Sionna RT alone |
| Z2 | 예 | 예 | Sionna RT alone, and it is exact |
| Z3 | 아니오 | 아니오 | Our PO surface integral, pattern only |
| Z4 | 아니오 | 예 | PO integral + measurement anchor |

출처 ⟨outputs/report00_decision_map.json : zones⟩

바닥 문장은 레이더 방정식 자신의 것이다 — The split is the radar equation's own: propagation is the engine's term, target scattering is a separate one. ⟨outputs/report00_decision_map.json : footer_en⟩

![report00 f4](outputs/figures/report00_f4.png)

**그림 4.** 두 물음을 던지면 각 실험은 어느 칸에 앉는가?

### 오른쪽 절반 — 표적 항이 소거되는 칸

| 실험 유형 | 칸 | 왜 그런지 |
|---|---|---|
| Chamber geometry, shadowing, occlusion | Z1 | 가림은 유한 기하로 정확히 판정된다 — 표적이 벽 뒤면 제대로 사라진다. |
| Multipath delay, floor-ghost timing | Z1 | 유령 경로의 지연은 경로 길이만으로 정해진다 — 표적 세기가 상수로 빠진다. |
| CFAR threshold from target-free noise | Z1 | 허위경보 문턱은 표적이 없는 셀의 통계로 정한다. 표적 항이 아예 등장하지 않는다. |
| Free-space link budget, absolute power | Z2 | 자유공간 직접파는 Friis 이론과 소수 일곱째 자리에서 일치한다. |
| Specular reflection strength off a wall | Z2 | 반사 세기는 Fresnel 4계수로 정확히 계산된다. 이미지-소스 해석해와 0.03% 안이다. |

출처 ⟨outputs/report00_decision_map.json : items⟩

### 왼쪽 절반 — 표적 항이 답에 남는 칸

| 실험 유형 | 칸 | 왜 그런지 |
|---|---|---|
| Aspect pattern shape vs azimuth | Z3 | 자세 패턴은 기하에서 나온다. 커널이 해석 PO 를 제대로 계산하는지가 관문이다. |
| Airframe-to-airframe ranking by shape | Z3 | 같은 재질·같은 정면면적에서도 모양만으로 31 dB 가 갈린다 — 반사계수로는 못 가른다. |
| Micro-Doppler modulation shape | Z3 | 부품별 회전 속도를 넣을 통로가 엔진에 없다. 위상을 가진 복소 산란장이 필요하다. |
| Absolute drone RCS in dBsm | Z4 | 절대 σ 는 기하 + 실측 앵커로만 선다. 불확도를 숨기지 않고 같이 인쇄한다. |
| Detection range and Pd benchmark | Z4 | 검출 거리는 σ 에 직접 걸린다. 크기를 40배 바꿔도 경로 진폭이 안 움직이는 도구로는 못 낸다. |
| Mesh fidelity budget for a drone target | Z4 | 실루엣이 유지돼도 정반사 채널만으로는 49 dB 가 무너진다 — 면적분이 있어야 메쉬 예산을 잴 수 있다. |

출처 ⟨outputs/report00_decision_map.json : items⟩

⚠ 단서 하나. 소거 논증은 표적이 **한 방향에서** 조명될 때의 것이다.

다중경로에서는 직접파와 바닥 반사가 서로 다른 방향에서 동시에 표적을 때리므로 표적 항이 방향마다 달라진다. 커널의 바이스태틱 일반형:  E ∝ Σ_hit |Γ| · e^{jk(û_i+û_s)·p} · d² ⟨outputs/report00_po_case.json : s2_our_kernel.derivation[7].statement⟩ 가 보여주듯 각 조명 방향마다 다른 E 가 나온다. 그때는 오른쪽 칸의 실험도 왼쪽으로 이사한다. ⭐ 이 표를 한 사례로 시험한 것이 **리포트 07 「마이크로도플러」** 다 — 도는 로터는 비율만 필요하므로 오른쪽 칸에 앉는다.

## §5. 그래서 왜 PO 인가 — 다섯 갈래의 지도

| 방법 | 무엇을 푸는가 |
|---|---|
| ① 완전파 (MoM / MLFMM / FDTD) | 맥스웰 방정식을 근사 없이 푼다. 크리핑파·다중산란·편파를 전부 포함한다. |
| ② SBR + PO (우리) | 광선으로 '어느 면이 실제로 조명되는가' 를 찾고, 그 면에서 PO 표면적분으로 σ 를 낸다. 상용 EM 솔버(FEKO/CST/HFSS SBR+)의 고주파 표준 방법이다. |
| ③ 통계 RCS 주입 (3GPP 표 조회) | σ 를 규격 표에서 읽어 각도 섹터로 조회하고 경로전력에 곱한다. |
| ④ 기하 대리표적 (큐브·박스·구) | 표적을 정육면체·직육면체·구로 바꾼다. 선행에서 가장 흔한 회피다. |
| ⑤ 실측 | 무향실·CATR 에서 직접 잰다. 절대 앵커의 최종 출처다. |

출처 ⟨outputs/report00_po_case.json : s1_alternatives.alternatives⟩

⭐ ②의 위치를 정확히 적는다. 그래픽 레이트레이서 위에 자기 PO 적분기를 얹는 것은 우리 발명이 아니라 **이 문제의 표준 대응**이고, 두 팀이 독립적으로 같은 곳에 도달했다. ⟨outputs/report00_po_case.json : s2_our_kernel.same_methodology_as.statement⟩

### 비용과 정확도 — 그리고 게재된 반론 하나

완전파는 정확도의 과녁이다. 우리 2D EFIE MoM 자체검사는 정확 원기둥 고유함수해 대비 0.00027 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.mom_selftest_worst_db⟩ 다.

그 대신 비용이 표를 못 만들게 한다. 게재본 문장 그대로 — it should be emphasized that MLFMM simulations are considerably more computationally demanding. For instance, each of the MLFMM simulations in this study took several hours. ⟨outputs/report00_po_case.json : s1_alternatives.alternatives[0].cost_quote_mlfmm⟩

우리 커널은 자세 하나(방위·고도 한 점 × 반송파 하나 → σ 한 값)에 중앙값 38.1 ms ⟨outputs/report00_po_case.json : s1_alternatives.ours_runtime.ours_per_pose_ms_median⟩ 다. 같은 `RTX 4090`, 같은 챔버 씬에서 스톡 `sionna.rt.PathSolver` 전파 해가 106.3 ms ⟨outputs/report00_po_case.json : s1_alternatives.stock_sionna_same_card.stock_sionna_ms_median⟩ 이므로 하드웨어 변수는 여기서 제거된다(⚠ 재는 양은 서로 다르다 — 전파 경로 대 σ).

⚠ 반론도 그대로 싣는다 — Ziganshin arXiv:2604.05991v2 pp.1–2, 게재된 반론: 'the need to cascade PO after RT negates the computational advantages of RT' ⟨outputs/report00_po_case.json : s1_alternatives.cascade_cost_objection._the_objection⟩

우리 구현에서 PO 적분은 광선캐스팅의 16.2 ⟨outputs/report00_po_case.json : s1_alternatives.cascade_cost_objection.our_po_over_rt⟩배다 — 적분이 아직 호스트 numpy 라서다.

게재된 유일한 GPU 커널 분해(SagittaSBR)는 같은 캐스케이드를 광선발사의 6.5% ⟨outputs/report00_po_case.json : s1_alternatives.cascade_cost_objection.sagitta_po_over_raylaunch_A100_fp32⟩ 로 적는다. 절반은 우리 몫이다.

## §6. 우리 PO 는 납득 가능한 수준인가 — 검증 3층과 자기검사 2건

분해가 먼저다. (커널 − 참값) = (커널 − 해석 PO) + (해석 PO − 참값). ①이 앞항을, ②③이 뒷항을 잰다. 두 항은 성질이 다르다 — 앞항은 **구현 오차**(격자를 조이면 준다), 뒷항은 **모형 오차**(격자로는 못 고친다). ⟨outputs/report00_po_case.json : s3_validation._the_decomposition⟩

| 층 | 과녁 | 무엇을 재나 | 결과 |
|---|---|---|---|
| ① | 해석 PO 구 (kr 1 ⟨outputs/report00_po_case.json : s3_validation.layer1_analytic_po_convergence.kr_sweep_kr_min⟩~100 ⟨outputs/report00_po_case.json : s3_validation.layer1_analytic_po_convergence.kr_sweep_kr_max⟩ · 입사 48 ⟨outputs/report00_po_case.json : s3_validation.layer1_analytic_po_convergence.kr_sweep_n_incidence⟩방향) | 커널 구현 | 최대 0.201 dB ⟨outputs/report00_po_case.json : s3_validation.layer1_analytic_po_convergence.kr_sweep_max_abs_db_vs_po_div16⟩ |
| ② | PEC 구 Mie 정확해 | PO 라는 모형 | ka=1 에서 -6.58 dB ⟨outputs/report00_po_case.json : s3_validation.layer2_pec_sphere_mie.po_minus_mie_at_ka1_db⟩, 격자 209 ⟨outputs/report00_po_case.json : s3_validation.layer1_analytic_po_convergence.sphere_ka1_grid_refine_factor⟩배 조여도 -0.054 dB ⟨outputs/report00_po_case.json : s3_validation.layer2_pec_sphere_mie.improvement_from_refining_grid_db⟩ 이동 · 광학영역 산포 1.83% ⟨outputs/report00_po_case.json : s3_validation.layer2_pec_sphere_mie.kr_sweep_std_pct_vs_mie_kr_ge30_div16⟩ |
| ③ | 얇은 띠 2D EFIE MoM | 가는 특징 | 가장 가는 시험 폭에서 TM -4.02 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.po_minus_tm_at_0p15lam_db⟩ · TE +7.53 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.po_minus_te_at_0p15lam_db⟩ (참값 자체가 편파로 11.56 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.tm_minus_te_at_0p15lam_db⟩ 갈린다) |
| + | PEC 이면각 닫힌형 8πa²b²/λ² | 다중반사 위상 | 2-bounce 최대 0.556 dB ⟨outputs/report00_po_case.json : s3_validation.layer4_dihedral_multibounce.max_abs_err_2bounce_db⟩ |
| + | 상반성 σ(û_i,û_s)=σ(û_s,û_i) | 정리 위반 = 모형오차 | 기체 최악 8.24 dB ⟨outputs/report00_po_case.json : s3_validation.layer5_reciprocity_selfcheck.drone_worst_violation_db⟩ (같은 검사를 인쇄한 선행 0편) |

### ⚠ 한계는 정면으로 — PO 무릎과 우리 세 밴드

무릎의 정의는 «max(|PO−MoM TM|,|PO−MoM TE|) 가 1 dB 를 넘는 구간의 상단 ⟨outputs/report00_po_case.json : s4_limits.po_validity_knee_rule⟩» 이고, 그 아래로 내려가려면 특징 폭이 0.729 ⟨outputs/report00_po_case.json : s4_limits.po_validity_knee_a_over_lambda⟩λ 이상이어야 한다. 그 무릎을 주파수로 옮기면 동체는 2.68 GHz ⟨outputs/report00_po_case.json : s4_limits.feature_knee_frequencies.body_81p51mm_ghz⟩, 팔뿌리는 4.86 GHz ⟨outputs/report00_po_case.json : s4_limits.feature_knee_frequencies.arm_root_45mm_ghz⟩, 블레이드는 15.86 GHz ⟨outputs/report00_po_case.json : s4_limits.feature_knee_frequencies.prop_blade_13p78mm_ghz⟩ 에서야 통과한다. 우리 생산 밴드는 LTE 1.843 GHz ⟨outputs/report00_po_case.json : s4_limits.our_production_bands_vs_knee.lte_ghz⟩ · 5G 3.5 GHz ⟨outputs/report00_po_case.json : s4_limits.our_production_bands_vs_knee.nr_ghz⟩ · WiFi 5.21 GHz ⟨outputs/report00_po_case.json : s4_limits.our_production_bands_vs_knee.wifi_ghz⟩ 이므로, **동체를 뺀 모든 특징이 무릎 아래에 있다.**

절대 σ 에는 격자 불확도도 함께 붙는다 — λ/16 서브셀 디더 산포 1.78 dB ⟨outputs/report00_po_case.json : s2_our_kernel.grid_dither.dither_spread_div16_db⟩ 다. 그리고 저대역 판정의 라벨 `B_PO_LIMIT ⟨outputs/report00_po_case.json : s4_limits.adversarial_verdict_verbatim.attacked_verdict_label⟩` 에 대한 적대검증 판정은 `PREMATURE ⟨outputs/report00_po_case.json : s4_limits.adversarial_verdict_verbatim.adversarial_verdict⟩` 다. 살아남은 것은 표본화 배제뿐이고, 정직한 라벨 세 장은 `A_EXCLUDED · B_CONTRIBUTES_SIGN_ONLY · C_UNBOUNDED` 다 — PO 한계는 **부호만** 확인됐고 크기 귀속은 아직 열려 있다.

부호는 한 방향을 가리킨다. 지배채널(TM) 기준 PO 는 얇은 특징을 **과소평가**한다(prop 0.083λ 에서 −7.25 dB). 따라서 우리 저주파 σ 는 낮게 나와 있을 개연성이 높고, 검출 성능 산출물은 **보수적(비관적)** 쪽으로 틀렸을 것이다. 단, 스칼라 PO 는 편파를 못 가르므로 TE 채널 기준으로는 부호가 반대다 — 방향을 못 박으려면 편파 있는 커널이 필요하다. ⟨outputs/report00_po_case.json : s4_limits.our_production_bands_vs_knee.sign_of_the_error⟩

## §7. 선행에서 무엇을 빌렸나

| 빌린 것 | 어디서 | 그것이 사 준 것 |
|---|---|---|
| 분포적합 · μ/ε 회귀 · 금속구 교정 · 분위점 · RMSE | Das(IEEE WCL 2026) · Yuan(EuCAP 2025) 이 실제로 쓴 절차 그대로 | report08 이 '보류(defer)' 로 남긴 절대 σ 판정을 문헌과 같은 자로 채웠다. |
| 금속구 교정 (calibration sphere) | Das 절차 + Zhang(JSAC 2026)의 0.5 m 금속구 잔차 검사 | 우리 커널이 생산 3 밴드에서 정확 Mie 대비 몇 dB 인지가 표가 됐다. |
| 정규화 RCS σ/(πr²) 축과 kr 유효성 바닥 | SagittaSBR(arXiv:2604.09243) — 독립 SBR+PO 솔버, 우리 최근친 | '우리가 몇 배 나쁘다' 는 옛 헤드라인이 애초에 다른 양이었음이 드러났고, kr≥30 에서 우리 산포가 그들과 같은 급이라는 것이 인쇄됐다. |
| 모노스태틱에서 출사 가시성 함수를 생략해도 되는 근거 | SagittaSBR 각주 1 (원문 인용이 src/rcs_sbr.py:538-542 주석에 그대로 실려 있다) | 바이스태틱에만 그림자광선을 1발 더 쏘는 설계의 근거. 모노에서 정확한 no-op 임을 실측으로 확인했다. |
| 표적 서명을 밖에서 만들어 경로에 곱하는 **주입 아키텍처** | IEEE TAP(Schuler 2008) → IET RSN(Deep 2020) 계보, 그리고 3GPP TR 38.901 7.9.2.1 | 우리 구조가 '발명' 이 아니라 '표준 구조' 라는 것. 기여 문장은 표를 만든다가 아니라 **두 번째 것**을 이름 붙여야 한다. |

출처 ⟨outputs/report00_po_case.json : s5_prior_work.already_borrowed⟩

### 아직 안 빌린 것 — 순위와 값

| 순위 | 무엇을 | 어디서 | 비용 |
|---|---|---|---|
| 1 | 세 유효성 바닥을 기체 × 밴드 표로 인쇄 | Sagitta(kr≥30 @2%, ds≤λ/5) · Monte-Carlo SBR(10 rays/λ) · Ziganshin(면 규칙 E>1.5λ, E²/(Rλ) 0.6–0.9) | 반나절, 새 실행 0 |
| 2 | 정준체(구·원통)를 우리 커널에 통과 + 캠페인 패드에 2.8 dBsm 원통 | Das 절차(IEEE Std 1502-2020) + Zhang 잔차 | 커널 1일 + 원통 가공비 |
| 3 | PTD-EEC 가산 모서리 항을 **진단으로** | Kirik & Özdemir 2019(SBR 에 PTD 를 꽂는 전체 알고리즘) + Öztürk 2002(유도·닫힌형, E_tot = E_PTDEEC + E_PO) | 1–2주 |
| 4 | SBR 마이크로도플러에 회전 불변성 적용 | jees2021(MoM) — 같은 연산에서 156.8× 를 쟀다 | 며칠 |
| 5 | 스칼라 γ_po 5개 → MECA 유전체 등가전류 | Monte-Carlo SBR (UMD, arXiv:2511.07586) | 1주 |
| 6 | 미분가능 재질 보정 | Hoydis 외(IEEE TMLCN 2024) — 경사하강 보정 4.93 → 2.16 → 1.00 dB | 큼. 실측 캠페인이 먼저다 |
| 7 | held-out 밴드 시험 | Zhang JSAC 2026 — 두 밴드로 정착시키고 세 번째를 예측 | 새 실행 1회 |

출처 ⟨outputs/report00_po_case.json : s5_prior_work.not_yet_borrowed⟩

맨 위 두 줄이 이 편의 다음 단계와 같다 — 유효성 바닥을 기체 × 밴드 표로 인쇄하는 일과, 정준체를 캠페인 패드에 올려 절대 σ 에 처음으로 실측 검사를 붙이는 일이다. 의도적으로 안 빌린 것도 셋이다 — «Sionna stock diffraction=True 를 표적에 켜기 ⟨outputs/report00_po_case.json : s5_prior_work.deliberately_not_borrowed[1].what⟩» 가 그중 하나이고, 이유는 도구가 다르다는 것이다: D 는 σ 가 아니라 경로계수를 만들고, wedge 각을 인접 두 면의 법선에서 읽으므로 면분할된 셸에서 그 각은 기체의 성질이 아니라 **우리 메싱의 성질**이 된다.

## §8. 아직 못 하는 것 — 열린 항목과 그 크기

| 열린 항목 | 현재 상태 | 크기 |
|---|---|---|
| 편파 | 면적분이 스칼라다 — 한 채널의 σ 를 낸다 | 가장 가는 시험 폭에서 참값이 TM−TE 11.56 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.tm_minus_te_at_0p15lam_db⟩ 로 갈린다 |
| 모서리 프린지(PTD) | 배선은 있고 생산 경로는 ptd=False 다 | PTD 를 켜면 TE 가 +7.53 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.po_minus_te_at_0p15lam_db⟩ → +10.81 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.po_ptd_minus_te_at_0p15lam_db⟩ 로 벌어진다 |
| 2회 이상 다중반사 | 생산 σ 는 1-bounce 다 | 이면각에서 1-bounce -118.33 dBsm ⟨outputs/report00_po_case.json : s3_validation.layer4_dihedral_multibounce.a03_sbr_1bounce_dbsm⟩ ↔ 2-bounce 14.51 dBsm ⟨outputs/report00_po_case.json : s3_validation.layer4_dihedral_multibounce.a03_sbr_2bounce_dbsm⟩ |
| 크리핑파·표면파 | 우리 커널과 Sionna 가 같은 자리에 선다 — GO/UTD 계열 고주파 근사의 바깥이다 | 매끄러운 볼록체 그림자 경계를 감아 도는 성분 |
| 전방산란 `β→180°` | 조명 게이트와 수신 게이트가 상호배타라 σ ≡ 0 이 된다 | 주장 창을 후방~중간 바이스태틱각으로 못박는다 |
| 생산 경로의 참값 대조 | 참값 앵커는 penetrate=False · \|Γ\|=1 · 볼록이다 | 생산은 penetrate=True · 재질 Γ · 자기가림 · 1-bounce 다 |
| 테셀레이션 축 | 메쉬 사다리는 앵커 물체 두 점에서 돌렸다 | 같은 평판을 쪼개기만 해도 9.54 dB ⟨outputs/report00_evidence.json : H_tessellation_changes_the_answer.numbers.max_inflation_db⟩ 부푼다 |
| 회전 프로펠러 도플러 | Sionna 의 Paths.doppler 는 객체당 강체 속도 1벡터다 | 부품별 위상은 우리 커널의 복소 E 에서 온다 |

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 공개된 세 유효성 바닥을 기체 × 밴드 표로 인쇄한다 | '전기적 소형' 이 고백에서 표가 되고, 어느 기체·어느 밴드가 공개 바닥 아래인지가 행 단위로 확정된다 | `benchmark/verify_sbr_kr_sweep.py` → 02편 §3 |
| 정준체(구·원통)를 캠페인 패드에 올려 우리 커널과 같은 자로 잰다 | 절대 σ 에 처음으로 실측 검사가 붙는다 — 레벨이 자체 앵커를 갖는다 | 06편 §2 측정 설계 |
| PO 면적분을 편파 있는 커널로 올린다 | 가장 가는 시험 폭에서 참값이 갈라지는 11.56 dB ⟨outputs/report00_po_case.json : s3_validation.layer3_thin_plate_2d_mom.tm_minus_te_at_0p15lam_db⟩ 가 우리 σ 의 어느 쪽 오차인지가 결정된다 | `src/rcs_sbr.py` → 이 편 §6 |
| 드론 본체에서 재테셀레이션 사다리를 돌린다 | 적대검증이 무경계로 남긴 기하 축(C)에 크기가 붙는다 | `benchmark/facet_mechanism.py` → 이 편 §3 |
| 다중경로 기하에서 표적 항이 소거되는지를 직접 잰다 | 결정표 오른쪽 칸의 실험이 챔버 안에서도 그 칸에 남는지가 확정된다 | 이 편 §4 결정표 → 05편 검출 결과 |
| PO 적분을 디바이스 커널로 내린다 | 캐스케이드 반론의 비율 16.2 ⟨outputs/report00_po_case.json : s1_alternatives.cascade_cost_objection.our_po_over_rt⟩배가 게재된 GPU 분해 수준으로 내려가는지가 결정된다 | `src/rcs_sbr.py` → 이 편 §5 |